In [0]:
%run ./01.config

In [0]:



from pyspark.sql import functions as F
from pyspark.sql.window import Window
import logging

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)


class SilverTransformer:
    
    def __init__(self, spark, source_table):
        self.spark = spark
        self.source_table = source_table
    
    def transform_and_save(self, target_table, rejected_table):
        try:
            logger.info(f"Loading {self.source_table}")
            df = self.spark.table(self.source_table)
            
            df = df.withColumns({
                "demand_quantity": F.regexp_replace("demand_quantity", ",", "").cast("double"),
                "available_inventory": F.regexp_replace("available_inventory", ",", "").cast("double"),
                "unit_price_usd": F.regexp_replace("unit_price_usd", ",", "").cast("double")
            })
            
            logger.info("Removing duplicates")
            window = Window.partitionBy("transaction_id").orderBy(F.col("_ingest_ts").desc())
            df = df.withColumn("_row_num", F.row_number().over(window))
            df = df.filter("_row_num = 1").drop("_row_num")
            
            logger.info("Quality checks")
            df = df.withColumn(
                "_dq_check",
                F.when(F.col("transaction_id").isNull(), "no_id")
                .when(F.col("demand_quantity").isNull(), "no_demand")
                .when(F.col("demand_quantity") < 0, "negative_demand")
                .otherwise("ok")
            )
            
            good = df.filter("_dq_check = 'ok'").drop("_dq_check", "_rescued_data")
            bad = df.filter("_dq_check != 'ok'")
            
            good.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)
            bad.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(rejected_table)
            
            good_count = good.count()
            bad_count = bad.count()
            
            logger.info(f"Done! Clean: {good_count:,} | Rejected: {bad_count:,}")
            return self.spark.table(target_table)
            
        except Exception as e:
            logger.error(f"Error: {str(e)}")
            raise


transformer = SilverTransformer(spark, BRONZE)
silver = transformer.transform_and_save(SILVER, SILVER_REJECTED)